In [2]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

def parse_cfg(path):
    """
    Parses the patient Info.cfg file.
    Returns a dictionary of clinical parameters and frame indices.
    """
    data = {}
    if not os.path.exists(path):
        return None
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            key, val = line.split(':', 1)
            data[key.strip()] = val.strip()
    return data

def analyze_dataset_splits(base_dir):
    """
    Scans the training and testing folders to compile clinical, 
    shape, and voxel-spacing metadata.
    """
    print("\n" + "="*50)
    print(" 1. SCANNING DATASET SPLITS & PATIENT METADATA")
    print("="*50)
    
    records = []
    
    for split in ['training', 'testing']:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            print(f"[Warning] Split folder not found: {split_dir}")
            continue
            
        patients = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
        patients.sort()
        
        print(f"Analyzing {split.upper()} split ({len(patients)} patients)...")
        
        for p in patients:
            p_dir = os.path.join(split_dir, p)
            cfg = parse_cfg(os.path.join(p_dir, "Info.cfg"))
            
            # Read 4D Cine volume dimensions using nibabel
            nii_4d_path = os.path.join(p_dir, f"{p}_4d.nii")
            h, w, d, t = "N/A", "N/A", "N/A", "N/A"
            spacing_x, spacing_y, spacing_z = "N/A", "N/A", "N/A"
            
            if os.path.exists(nii_4d_path):
                try:
                    img = nib.load(nii_4d_path)
                    shape = img.shape
                    if len(shape) == 4:
                        h, w, d, t = shape
                    elif len(shape) == 3:
                        h, w, d = shape
                        t = 1
                    
                    zooms = img.header.get_zooms()
                    spacing_x, spacing_y = round(zooms[0], 3), round(zooms[1], 3)
                    if len(zooms) >= 3:
                        spacing_z = round(zooms[2], 3)
                except Exception as e:
                    print(f"  [Error] Failed to load {nii_4d_path}: {e}")
            
            records.append({
                "Patient ID": p,
                "Split": split.capitalize(),
                "Pathology Group": cfg.get("Group", "N/A") if cfg else "N/A",
                "Resolution (H x W)": f"{h} x {w}",
                "Slices (D)": d,
                "Frames (T)": t,
                "Voxel Spacing X (mm)": spacing_x,
                "Voxel Spacing Y (mm)": spacing_y,
                "Slice Thickness Z (mm)": spacing_z,
                "ED Frame": int(cfg.get("ED", 0)) if cfg and cfg.get("ED") else "N/A",
                "ES Frame": int(cfg.get("ES", 0)) if cfg and cfg.get("ES") else "N/A",
                "Weight (kg)": float(cfg.get("Weight", 0)) if cfg and cfg.get("Weight") else "N/A",
                "Height (cm)": float(cfg.get("Height", 0)) if cfg and cfg.get("Height") else "N/A"
            })
            
    df = pd.DataFrame(records)
    
    # Print Quick Aggregations
    for split in ['Training', 'Testing']:
        split_df = df[df['Split'] == split]
        if split_df.empty:
            continue
        print(f"\n--- {split.upper()} SPLIT SUMMARY ---")
        print(f"Pathology Groups: {split_df['Pathology Group'].value_counts().to_dict()}")
        print(f"Slice Count Range (Depth D): Min={split_df['Slices (D)'].min()}, Max={split_df['Slices (D)'].max()}")
        print(f"Frame Count Range (Time T): Min={split_df['Frames (T)'].min()}, Max={split_df['Frames (T)'].max()}")
        print(f"Average Height: {split_df['Height (cm)'].mean():.1f} cm | Average Weight: {split_df['Weight (kg)'].mean():.1f} kg")
        
    return df

def analyze_voxel_labels(base_dir, sample_size=15):
    """
    Loads ground truth segmentations from a sample of patients to calculate
    voxel class distributions.
    """
    print("\n" + "="*50)
    print(" 2. DETAILED VOXEL & SEGMENTATION LABEL DISTRIBUTION")
    print("="*50)
    
    train_dir = os.path.join(base_dir, 'training')
    patients = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))][:sample_size]
    
    label_counts = {0: 0, 1: 0, 2: 0, 3: 0}
    unique_labels = set()
    
    print(f"Profiling segmentation label distribution (Sampling first {sample_size} training patients)...")
    
    for p in patients:
        p_dir = os.path.join(train_dir, p)
        cfg = parse_cfg(os.path.join(p_dir, "Info.cfg"))
        if not cfg:
            continue
        
        ed_str = cfg.get("ED").zfill(2)
        es_str = cfg.get("ES").zfill(2)
        
        for frame_suffix in [f"frame{ed_str}", f"frame{es_str}"]:
            gt_path = os.path.join(p_dir, f"{p}_{frame_suffix}_gt.nii")
            if os.path.exists(gt_path):
                img = nib.load(gt_path)
                data = img.get_fdata()
                uniques, counts = np.unique(data, return_counts=True)
                for u, c in zip(uniques, counts):
                    val = int(round(u))
                    unique_labels.add(val)
                    if val in label_counts:
                        label_counts[val] += c
                        
    total_voxels = sum(label_counts.values())
    if total_voxels > 0:
        print("\nLabel Statistics:")
        print(f"  Unique Labels Found: {sorted(list(unique_labels))}")
        print(f"  Voxel Class Balance:")
        print(f"    - Class 0 (Background):  {label_counts[0]:12d} voxels ({label_counts[0]/total_voxels*100:6.2f}%)")
        print(f"    - Class 1 (RV Cavity):   {label_counts[1]:12d} voxels ({label_counts[1]/total_voxels*100:6.2f}%)")
        print(f"    - Class 2 (Myocardium):  {label_counts[2]:12d} voxels ({label_counts[2]/total_voxels*100:6.2f}%)")
        print(f"    - Class 3 (LV Cavity):   {label_counts[3]:12d} voxels ({label_counts[3]/total_voxels*100:6.2f}%)")

def save_visualizations(base_dir, output_dir):
    """
    Renders a side-by-side plot of the raw slice, mask, and overlay
    for the first patient and saves the image to disk.
    """
    print("\n" + "="*50)
    print(" 3. GENERATING SAMPLE SEGMENTATION PREVIEW")
    print("="*50)
    
    patient_id = "patient001"
    p_dir = os.path.join(base_dir, "training", patient_id)
    img_path = os.path.join(p_dir, f"{patient_id}_frame01.nii")
    gt_path = os.path.join(p_dir, f"{patient_id}_frame01_gt.nii")
    
    if not os.path.exists(img_path) or not os.path.exists(gt_path):
        print("  [Error] patient001 files not found for visualization.")
        return
        
    img = nib.load(img_path).get_fdata()
    gt = nib.load(gt_path).get_fdata()
    
    slice_idx = img.shape[2] // 2  # Extract middle slice
    
    img_slice = img[:, :, slice_idx]
    gt_slice = gt[:, :, slice_idx]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Raw MRI
    axes[0].imshow(img_slice, cmap='gray')
    axes[0].set_title(f"Raw MRI (Slice {slice_idx})")
    axes[0].axis('off')
    
    # 2. Ground Truth Mask
    axes[1].imshow(gt_slice, cmap='tab10', vmin=0, vmax=3)
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis('off')
    
    # 3. Overlaid View
    axes[2].imshow(img_slice, cmap='gray')
    masked_gt = np.ma.masked_where(gt_slice == 0, gt_slice)
    axes[2].imshow(masked_gt, cmap='rainbow', alpha=0.5, vmin=1, vmax=3)
    axes[2].set_title("Overlay (Purple=RV, Green=MYO, Red=LV)")
    axes[2].axis('off')
    
    plt.tight_layout()
    
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, "acdc_sample_visualization.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"Sample visualization exported to: {save_path}")

if __name__ == "__main__":
    # Base directory containing the 'training' and 'testing' folders
    ACDC_BASE_DIR = r"C:\Users\NITRO V 15\Downloads\archive (3)\database"
    
    # Directory where you want to output report files and images
    OUTPUT_REPORT_DIR = r"C:\D\ACDC\data-exploration"
    
    print("Starting ACDC Dataset Full Exploration Workflow...")
    
    # 1. Analyze splits and clinical info
    df_inventory = analyze_dataset_splits(ACDC_BASE_DIR)
    
    # Save the dataframe summary as CSV
    csv_out = os.path.join(OUTPUT_REPORT_DIR, "acdc_dataset_inventory.csv")
    os.makedirs(OUTPUT_REPORT_DIR, exist_ok=True)
    df_inventory.to_csv(csv_out, index=False)
    print(f"Complete dataset inventory saved to CSV at: {csv_out}")
    
    # 2. Analyze voxel distribution
    analyze_voxel_labels(ACDC_BASE_DIR, sample_size=15)
    
    # 3. Save visualizations
    save_visualizations(ACDC_BASE_DIR, OUTPUT_REPORT_DIR)
    
    print("\nExploration completed successfully!")

Starting ACDC Dataset Full Exploration Workflow...

 1. SCANNING DATASET SPLITS & PATIENT METADATA
Analyzing TRAINING split (100 patients)...
Analyzing TESTING split (50 patients)...

--- TRAINING SPLIT SUMMARY ---
Pathology Groups: {'DCM': 20, 'HCM': 20, 'MINF': 20, 'NOR': 20, 'RV': 20}
Slice Count Range (Depth D): Min=6, Max=18
Frame Count Range (Time T): Min=12, Max=35
Average Height: 170.8 cm | Average Weight: 75.0 kg

--- TESTING SPLIT SUMMARY ---
Pathology Groups: {'DCM': 10, 'NOR': 10, 'MINF': 10, 'HCM': 10, 'RV': 10}
Slice Count Range (Depth D): Min=6, Max=21
Frame Count Range (Time T): Min=14, Max=35
Average Height: 170.0 cm | Average Weight: 80.7 kg
Complete dataset inventory saved to CSV at: C:\D\ACDC\data-exploration\acdc_dataset_inventory.csv

 2. DETAILED VOXEL & SEGMENTATION LABEL DISTRIBUTION
Profiling segmentation label distribution (Sampling first 15 training patients)...

Label Statistics:
  Unique Labels Found: [0, 1, 2, 3]
  Voxel Class Balance:
    - Class 0 (Back